# Week 3 — Ask for a qualitative suggestion in JSON

**Research task:** Read four short interview excerpts, write your own first memo, ask for a provisional theme and source ID, then check contrasting passages and record a revised claim.

**Python introduced:** a route selector, `if/else`, lists of dictionaries, `json.dumps(...)` and `json.loads(...)`.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session03/session03_qualitative_interpretation.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the Python SDKs used by this notebook, downloads the public course repository and selects OpenRouter because Colab cannot reach the Ollama server on your computer. On a local machine it installs nothing silently: it checks that the notebook is using the course environment and gives the exact repair command if it is not.

The setup also adds the repository root to Python's import path. This is necessary for Week 4's supplied image utility and prevents a second kind of `ModuleNotFoundError` after the repository has been cloned.

For local work, download the complete repository rather than this notebook alone, start it with `uv run jupyter lab`, and follow any `NEXT STEP` printed by the setup cell. The full instructions are in `docs/ENVIRONMENT_SETUP.md` and in the course book's computing chapter.


In [ ]:
SESSION = "session03"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import importlib.util as setup_importlib_util
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The complete GenAI_Soc2026 repository could not be found. A notebook "
            "downloaded by itself is not enough for local work. Download or clone the "
            "repository, open a terminal in that folder, and run: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib_util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Python executable:", setup_sys.executable)
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")
else:
    import json as setup_json
    import ollama as setup_ollama

    setup_config = setup_json.loads(
        (COURSE_ROOT / "config" / "course_models.json").read_text()
    )
    setup_local_model = setup_config["local"]["model"]
    try:
        setup_models = setup_ollama.list().models
        setup_model_names = [
            getattr(item, "model", None) or getattr(item, "name", None)
            for item in setup_models
        ]
        print("Ollama server: reachable at localhost:11434")
        if setup_local_model in setup_model_names:
            print("Course local model: ready —", setup_local_model)
        else:
            print("Course local model: NOT INSTALLED —", setup_local_model)
            print("NEXT STEP: open a terminal and run: ollama pull " + setup_local_model)
    except Exception as setup_error:
        print("Ollama server: NOT REACHABLE")
        print("NEXT STEP: start the Ollama application, then run: ollama list")
        print("Diagnostic:", str(setup_error).splitlines()[0])


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Choose one route and read four source excerpts

`ROUTE` is a string controlling which branch runs. `excerpts` is a list of four dictionaries; each keeps a source ID beside its text. `T01_A` and `T01_B` are two moments in one fictional tenant's account. `T02_A` and `T03_A` complicate a simple story of help becoming political action. These are instructor-authored synthetic passages.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.


In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # Ollama is the local default
if ROUTE == "openrouter" and not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")
excerpts = [
    {"id": "T01_A", "text": "The grocery deliveries helped. At first I did not know who organized them."},
    {"id": "T01_B", "text": "Months later I helped with childcare and joined the tenants' meeting."},
    {"id": "T02_A", "text": "I accepted help but kept it separate from politics. I never attended meetings."},
    {"id": "T03_A", "text": "When the delivery rota became an obligation, I left the group."},
]
print("Route:", ROUTE)
print("First encounter:", excerpts[0])
print("Later account from the same tenant:", excerpts[1])
print("Contrasting accounts:", excerpts[2], excerpts[3])


## Write your first reading before the model call

**Input:** your reading of the four excerpts. **Output:** `first_memo`, a string. Save it before seeing a model suggestion so you can later ask whether the suggestion redirected your attention.


In [ ]:
first_memo = "Practical help sometimes led to meetings, but not for everyone."
print("My first reading:", first_memo)

## Convert the excerpts into prompt text and construct messages

`json.dumps(excerpts)` converts the source list to text without losing IDs. The prompt gives the research question, task and required fields. It does **not** include `first_memo`, so the model cannot see your independent first reading. `messages` is the list sent to the SDK.


In [ ]:
prompt = (
    "Research question: How do tenants distinguish emergency help from political solidarity? "
    "Suggest one provisional theme. Cite one excerpt ID that supports it. "
    "Return JSON with exactly theme, evidence_id, and question_for_researcher. "
    "Do not treat a theme as a finding. Excerpts: " + json.dumps(excerpts)
)
messages = [{"role": "user", "content": prompt}]
print(prompt)

## Make the selected route's call without hiding either branch

`if ROUTE == "openrouter"` selects the hosted call; `else` selects the local call. Both assign returned text to `raw_output`. `temperature=0` asks for less variation; it does not make the interpretation correct or guarantee identical reruns.


In [ ]:
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(
            model=HOSTED_MODEL,
            messages=messages,
            temperature=0,
            response_format={"type": "json_object"},
        )
    raw_output = response.choices[0].message.content
else:
    response = ollama.chat(think=False, 
        model=LOCAL_MODEL,
        messages=messages,
        format="json",
        options={"temperature": 0},
    )
    raw_output = response.message.content

print("Raw JSON text:", raw_output)


## Parse the JSON string and return to its cited source

`json.loads(raw_output)` turns the returned string into a dictionary. Square brackets retrieve the theme and cited ID. The short `for` loop looks for that ID in the four supplied records. If `cited_excerpt` is `None`, the cited ID does not exist. If it exists, read the passage: existence alone does not establish support.


In [ ]:
suggestion = json.loads(raw_output)
evidence_id = suggestion["evidence_id"]
cited_excerpt = None
for excerpt in excerpts:
    if excerpt["id"] == evidence_id:
        cited_excerpt = excerpt

print("Theme:", suggestion["theme"])
print("Cited excerpt:", cited_excerpt)
print("Question:", suggestion["question_for_researcher"])

## Compare with two contrasting passages

**Input:** the model suggestion, your earlier memo, the cited record and two other excerpts. **Output:** printed evidence for your own review. `is not None` checks only whether the cited ID exists; it does not interpret the passage. Ask what the suggestion made you notice or overlook.


In [ ]:
countercase_one = excerpts[2]
countercase_two = excerpts[3]
print("The model suggested:", suggestion["theme"])
print("I wrote before the call:", first_memo)
print("Does the cited ID exist?", cited_excerpt is not None)
print("Contrasting case 1:", countercase_one)
print("Contrasting case 2:", countercase_two)

## Record a researcher revision

The wording below is an **illustrative researcher decision**, not an automatic answer. `analysis_record` is a dictionary holding the first memo, route, request, raw return, parsed suggestion, cited source, countercase IDs and the researcher's reason. After reading the passages, explain where you agree or disagree with this example.


In [ ]:
revised_claim = (
    "Repeated exchanges sometimes led to meeting participation, "
    "but receiving help did not always produce political involvement."
)
researcher_reason = (
    "T01_B follows repeated contact; T02_A separates help from politics; "
    "T03_A describes withdrawal when help felt obligatory."
)
analysis_record = {
    "first_memo": first_memo,
    "route": ROUTE,
    "model_request": prompt,
    "raw_model_output": raw_output,
    "model_suggestion": suggestion,
    "cited_excerpt": cited_excerpt,
    "countercase_ids": ["T02_A", "T03_A"],
    "revised_claim": revised_claim,
    "researcher_reason": researcher_reason,
}
print("Revised claim:", analysis_record["revised_claim"])
print("Why it changed:", analysis_record["researcher_reason"])

## Methodological check

Edit only `T02_A` in the source-list cell to: `The food deliveries helped, but I avoided the group because meetings felt hostile.` Rerun from the prompt cell onward. Compare the new raw return with the first one, reopen the cited passage and both contrasting cases, then say whether your revised claim should change. Well-formed JSON does not decide the interpretation.
## Completion recording

Use one chosen route and change only `T02_A`. Show the first memo, model suggestion, cited passage, two contrasting cases and your own revised claim. Explain each input and output, the route branch that ran, how the source lookup works and what changed between passes.

Explain every input and output aloud. Never show the shared key.